# Phase 2: Tabular XGBoost Baseline

Train a baseline XGBoost model on raw tabular features only.

In [1]:
import os
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb

DATA_PATH = "/Users/shengqu/Desktop/CSCI5253/FinalProject/ieee-fraud-detection/data/train_merged.parquet" 
MODEL_OUT  = "/Users/shengqu/Desktop/CSCI5253/FinalProject/model_output/xgb_baseline.json"

/Users/shengqu/opt/anaconda3/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/Users/shengqu/opt/anaconda3/lib/python3.9/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


## 1. Load Data

In [2]:
df = pd.read_parquet(DATA_PATH)
print(f"Shape: {df.shape}")
df.head(3)

Shape: (590540, 434)


,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,...,id_31,id_32,id_33,id_34,id_35,id_36,id_37,id_38,DeviceType,DeviceInfo
0,2987000,0,86400,68.5,W,13926,NaN,150.0,discover,142.0,...,None,NaN,None,None,None,None,None,None,None,None
1,2987001,0,86401,29.0,W,2755,404.0,150.0,mastercard,102.0,...,None,NaN,None,None,None,None,None,None,None,None
2,2987002,0,86469,59.0,W,4663,490.0,150.0,visa,166.0,...,None,NaN,None,None,None,None,None,None,None,None


## 2. Time-Based Train / Validation Split

In [3]:
df = df.sort_values("TransactionDT").reset_index(drop=True)
split_idx = int(len(df) * 0.8)
train, valid = df.iloc[:split_idx], df.iloc[split_idx:]

print(f"Train : {len(train):,} rows  |  fraud rate: {train['isFraud'].mean():.4%}")
print(f"Valid : {len(valid):,} rows  |  fraud rate: {valid['isFraud'].mean():.4%}")

Train : 472,432 rows  |  fraud rate: 3.5135%
Valid : 118,108 rows  |  fraud rate: 3.4409%


## 3. Feature Selection

In [4]:
drop_cols = ["isFraud", "TransactionID", "TransactionDT"]
y_train, y_valid = train["isFraud"], valid["isFraud"]
X_train = train.drop(columns=drop_cols).copy()
X_valid = valid.drop(columns=drop_cols).copy()

print(f"Starting feature count: {X_train.shape[1]}")

Starting feature count: 431


### 3a. Drop Near-Empty Columns 

In [5]:
# Compute missingness on the training set only (never peek at valid)
miss_rate = X_train.isnull().mean()

drop_high_miss = miss_rate[miss_rate > 0.90].index.tolist()
X_train.drop(columns=drop_high_miss, inplace=True)
X_valid.drop(columns=drop_high_miss, inplace=True)

print(f"Dropped {len(drop_high_miss)} columns with >90% missing:")
print(drop_high_miss)

Dropped 12 columns with >90% missing:
['dist2', 'D7', 'id_07', 'id_08', 'id_18', 'id_21', 'id_22', 'id_23', 'id_24', 'id_25', 'id_26', 'id_27']


### 3b. Add Missing Indicator Columns

In [6]:
# Recompute miss_rate after dropping the >90% columns
miss_rate = X_train.isnull().mean()
mid_miss_cols = miss_rate[(miss_rate > 0.50) & (miss_rate <= 0.90)].index.tolist()

for c in mid_miss_cols:
    X_train[f"{c}_missing"] = X_train[c].isnull().astype(np.int8)
    X_valid[f"{c}_missing"] = X_valid[c].isnull().astype(np.int8)

print(f"Added {len(mid_miss_cols)} missingness indicator columns for features with 50–90% missing.")
print("Examples:", mid_miss_cols[:10])

/var/folders/ws/c4n8v3rs1_g18bkf6s3m1lhc0000gn/T/ipykernel_36163/2967380334.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X_train[f"{c}_missing"] = X_train[c].isnull().astype(np.int8)
/var/folders/ws/c4n8v3rs1_g18bkf6s3m1lhc0000gn/T/ipykernel_36163/2967380334.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X_valid[f"{c}_missing"] = X_valid[c].isnull().astype(np.int8)
/var/folders/ws/c4n8v3rs1_g18bkf6s3m1lhc0000gn/T/ipykernel_36163/2967380334.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the

Added 217 missingness indicator columns for features with 50–90% missing.
Examples: ['dist1', 'R_emaildomain', 'D5', 'D6', 'D8', 'D9', 'D11', 'D12', 'D13', 'D14']


/var/folders/ws/c4n8v3rs1_g18bkf6s3m1lhc0000gn/T/ipykernel_36163/2967380334.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X_train[f"{c}_missing"] = X_train[c].isnull().astype(np.int8)
/var/folders/ws/c4n8v3rs1_g18bkf6s3m1lhc0000gn/T/ipykernel_36163/2967380334.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X_valid[f"{c}_missing"] = X_valid[c].isnull().astype(np.int8)
/var/folders/ws/c4n8v3rs1_g18bkf6s3m1lhc0000gn/T/ipykernel_36163/2967380334.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the

### 3c. Label-Encode

In [7]:
for c in X_train.select_dtypes(include="object").columns:
    le = LabelEncoder()
    combined = pd.concat([X_train[c], X_valid[c]], axis=0).astype(str).fillna("NA")
    le.fit(combined)
    X_train[c] = le.transform(X_train[c].astype(str).fillna("NA"))
    X_valid[c] = le.transform(X_valid[c].astype(str).fillna("NA"))

print(f"Final feature count: {X_train.shape[1]}")

Final feature count: 636


## 4. Train XGBoost

In [ ]:
model = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method="hist",
    eval_metric="auc",
    early_stopping_rounds=50,
    n_jobs=-1,
)
model.fit(X_train, y_train, eval_set=[(X_valid, y_valid)], verbose=50)

[0]	validation_0-auc:0.71109
[50]	validation_0-auc:0.85573


## 5. Evaluate

In [ ]:
p = model.predict_proba(X_valid)[:, 1]
roc = roc_auc_score(y_valid, p)
pr  = average_precision_score(y_valid, p)

print(f"Validation ROC-AUC : {roc:.4f}")
print(f"Validation PR-AUC  : {pr:.4f}   (naive baseline = {y_valid.mean():.4f})")
print()

Validation ROC-AUC : 0.9216
Validation PR-AUC  : 0.5854   (naive baseline = 0.0344)

Record these numbers — this is the benchmark to beat with graph features.


## 6. Save Model

In [ ]:
model.save_model(MODEL_OUT)
print(f"Saved baseline model to {MODEL_OUT}")

Saved baseline model to /Users/shengqu/Desktop/CSCI5253/FinalProject/model_output/xgb_baseline.json
Next step: 03_build_uid.ipynb
